# AbdiSQL — Turnkey Training & Evaluation Pipeline

**ABDI Platform — Model #1: AbdiSQL**  

This notebook executes the locked build plan for **AbdiSQL v1**:
- **Dialect:** SQLite
- **Base Model:** `Qwen/Qwen2.5-Coder-1.5B-Instruct` (fallback: `0.5B`)
- **Method:** QLoRA 4-bit (`bitsandbytes` + `peft` + `trl`)
- **Hardware:** Google Colab T4 GPU (free tier)
- **Dataset:** Spider cross-database text-to-SQL (1000–2000 balanced subset)
- **Evaluation:** Automated SQLite Execution Accuracy against ground truth

---

## Phase 1: Environment Setup

1. Mount Google Drive to preserve datasets, checkpoints, and merged weights across Colab session disconnects.
2. Verify Tesla T4 GPU and CUDA visibility.
3. Install pinned dependencies.

In [2]:
# 1.1 Mount Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')

# Create project directory on Drive
DRIVE_DIR = '/content/drive/MyDrive/abdi/abdisql'
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/data", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/merged_model", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/eval_results", exist_ok=True)
print(f"Project workspace initialized at: {DRIVE_DIR}")

Mounted at /content/drive
Project workspace initialized at: /content/drive/MyDrive/abdi/abdisql


In [3]:
# 1.2 Verify T4 GPU Availability
!nvidia-smi

import torch
assert torch.cuda.is_available(), "CUDA not visible to PyTorch — stop and switch Runtime to T4 GPU!"
gpu_name = torch.cuda.get_device_name(0)
print(f"Active GPU: {gpu_name}")
assert "T4" in gpu_name or "Tesla" in gpu_name or "A100" in gpu_name or "V100" in gpu_name, \
    f"Unexpected GPU runtime: {gpu_name}. Ensure T4 is selected."

Sun Sep 20 04:18:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
# 1.3 Install Required Packages
# Upgrade bitsandbytes to latest to avoid 'No module named triton.ops' on newer Python/Triton runtimes
!pip install -q --upgrade pip
!pip install -q -U bitsandbytes transformers peft trl accelerate datasets gdown
print("All dependencies installed and upgraded successfully.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 22.6 MB/s eta 0:00:00
All dependencies installed and upgraded successfully.


## Phase 2: Base Model Selection & Gate Check

1. Load `Qwen/Qwen2.5-Coder-1.5B-Instruct` in 4-bit.
2. Test 5–10 zero-shot SQL generation prompts.
3. **Gate check:** If OOM or unworkably slow, automatically switch to `Qwen/Qwen2.5-Coder-0.5B-Instruct`.
4. Save zero-shot baseline outputs — this is the mandatory comparison benchmark for Phase 6.

In [5]:
import torch
import time
import json
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Set to True if you want to explicitly use the lightweight 0.5B model instead of 1.5B
USE_05B_MODEL = False

BASE_MODEL_ID = "Qwen/Qwen2.5-Coder-0.5B-Instruct" if USE_05B_MODEL else "Qwen/Qwen2.5-Coder-1.5B-Instruct"
FALLBACK_MODEL_ID = "Qwen/Qwen2.5-Coder-0.5B-Instruct"

# 4-bit Quantization Config optimized for Tesla T4 (FP16 compute)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

def load_base_model(model_id):
    print(f"Loading {model_id} in 4-bit...")
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16,
        trust_remote_code=True
    )
    return tokenizer, model

try:
    tokenizer, model = load_base_model(BASE_MODEL_ID)
    ACTIVE_MODEL_ID = BASE_MODEL_ID
    print(f"Successfully loaded primary model: {ACTIVE_MODEL_ID}")
except Exception as e:
    print(f"Primary model failed with error: {e}\nFalling back to {FALLBACK_MODEL_ID}...")
    tokenizer, model = load_base_model(FALLBACK_MODEL_ID)
    ACTIVE_MODEL_ID = FALLBACK_MODEL_ID
    print(f"Active model set to fallback: {ACTIVE_MODEL_ID}")

# Report memory usage
allocated = torch.cuda.memory_allocated(0) / 1024**3
reserved = torch.cuda.memory_reserved(0) / 1024**3
print(f"VRAM Allocated: {allocated:.2f} GB | Reserved: {reserved:.2f} GB")


Loading Qwen/Qwen2.5-Coder-1.5B-Instruct in 4-bit...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Successfully loaded primary model: Qwen/Qwen2.5-Coder-1.5B-Instruct
VRAM Allocated: 1.07 GB | Reserved: 1.12 GB


In [6]:
# 2.2 Run 5-10 Zero-Shot Verification Prompts
SAMPLE_PROMPTS = [
    {
        "schema": "Database Schema: company\nTable employees: (emp_id number (PRIMARY KEY), name text, department text, salary number)",
        "question": "Find the names of all employees in the 'Sales' department who earn more than 50000."
    },
    {
        "schema": "Database Schema: school\nTable students: (id number (PRIMARY KEY), name text, grade number)\nTable courses: (course_id number, student_id number, course_name text)",
        "question": "How many students are enrolled in 'Math'?"
    },
    {
        "schema": "Database Schema: library\nTable books: (book_id number (PRIMARY KEY), title text, author text, year number)",
        "question": "List all book titles published after 2020 ordered by year descending."
    }
]

SYSTEM_PROMPT = "You are AbdiSQL, a domain-expert language model specialized in generating precise SQLite queries from database schemas and questions."

# Warmup pass to compile CUDA kernels and eliminate one-time cold-start latency
print("Warming up CUDA kernels...")
warmup_inputs = tokenizer("SELECT 1;", return_tensors="pt").to("cuda")
with torch.no_grad():
    _ = model.generate(**warmup_inputs, max_new_tokens=10)
print("Warmup complete. Running zero-shot generation gate check...")

latencies = []
for idx, item in enumerate(SAMPLE_PROMPTS):
    user_msg = f"{item['schema']}\n\nQuestion: {item['question']}"
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_msg}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    t0 = time.time()
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=128, do_sample=False)
    latency = time.time() - t0
    latencies.append(latency)

    prompt_len = inputs["input_ids"].shape[1]
    gen_text = tokenizer.decode(output[0][prompt_len:], skip_special_tokens=True).strip()
    print(f"\n[Prompt {idx+1}] Latency: {latency:.2f}s")
    print(f"Generated: {gen_text}")

avg_latency = sum(latencies) / len(latencies)
print(f"\nAverage Latency per Generation: {avg_latency:.2f}s")

# Gate Check: Ensure generation runs in a few seconds (not minutes / CPU paging)
assert avg_latency < 20.0, f"Average generation latency too slow ({avg_latency:.2f}s)! Switch USE_05B_MODEL = True."
print("\nPhase 2 Gate Check: PASSED. Model runs smoothly within T4 latency and VRAM limits.")


Warming up CUDA kernels...
Warmup complete. Running zero-shot generation gate check...

[Prompt 1] Latency: 13.85s
Generated: Here is the SQL query that will generate the desired result:

```sql
SELECT name FROM employees WHERE department = 'Sales' AND salary > 50000;
```

This query selects the `name` column from the `employees` table where the `department` is 'Sales' and the `salary` is greater than 50000. The `WHERE` clause filters the rows based on these conditions, and the `SELECT` statement retrieves the names of the matching employees.

[Prompt 2] Latency: 2.34s
Generated: SELECT COUNT(student_id) FROM courses WHERE course_name = 'Math'

[Prompt 3] Latency: 8.44s
Generated: Here is the SQL query that corresponds to your question:

```sql
SELECT title FROM books WHERE year > 2020 ORDER BY year DESC;
```

This query selects the `title` column from the `books` table where the `year` is greater than 2020. The results are then ordered by the `year` column in descending order.

Averag

## Phase 3: Dataset Preparation

1. Download Spider dataset (Spider train + tables + SQLite databases).
2. Sample 1000–2000 examples with balanced coverage across database schemas (`db_id`).
3. Split: ~90% train / ~10% held-out eval (minimum 100 eval examples).
4. Save directly to Google Drive so data survives session disconnects.

In [7]:
# 3.1 Download and Extract Spider Dataset (Robust Hugging Face mirror with complete SQLite databases)
import os
import zipfile
import shutil

SPIDER_ZIP = "spider_data.zip"
DRIVE_SPIDER_ZIP = f"{DRIVE_DIR}/spider_data.zip"
SPIDER_HF_URL = "https://huggingface.co/datasets/HAL-9001/spider-databases/resolve/main/spider_data.zip"

if not (os.path.exists("spider/train_spider.json") and os.path.exists("spider/tables.json") and os.path.exists("spider/database")):
    if os.path.exists(DRIVE_SPIDER_ZIP):
        print(f"Found cached Spider archive on Google Drive: {DRIVE_SPIDER_ZIP}")
        shutil.copy(DRIVE_SPIDER_ZIP, SPIDER_ZIP)
    else:
        print(f"Downloading Spider dataset archive (~196MB) from Hugging Face mirror...")
        !wget -q --show-progress -O spider_data.zip "{SPIDER_HF_URL}"
        # Cache to Drive so future sessions don't need to re-download
        try:
            shutil.copy(SPIDER_ZIP, DRIVE_SPIDER_ZIP)
            print(f"Cached Spider archive to Drive at: {DRIVE_SPIDER_ZIP}")
        except Exception as e:
            print(f"Drive cache note: {e}")

    print("Extracting Spider archive...")
    with zipfile.ZipFile(SPIDER_ZIP, 'r') as zip_ref:
        zip_ref.extractall(".")

    # Standardize folder name: if extracted folder is named spider_data, rename to spider
    if os.path.exists("spider_data") and not os.path.exists("spider"):
        os.rename("spider_data", "spider")
    elif os.path.exists("spider_data") and os.path.exists("spider"):
        shutil.rmtree("spider")
        os.rename("spider_data", "spider")

    print("Spider dataset extraction complete!")
    print(f" - tables.json exists: {os.path.exists('spider/tables.json')}")
    print(f" - train_spider.json exists: {os.path.exists('spider/train_spider.json')}")
    print(f" - database directory exists: {os.path.exists('spider/database')}")
else:
    print("Spider dataset already verified and ready in ./spider directory.")


Found cached Spider archive on Google Drive: /content/drive/MyDrive/abdi/abdisql/spider_data.zip
Extracting Spider archive...
Spider dataset extraction complete!
 - tables.json exists: True
 - train_spider.json exists: True
 - database directory exists: True


In [8]:
# 3.2 Process Schema and Build Balanced Dataset Subset
from collections import defaultdict
import random

def format_schema(db_schema):
    db_id = db_schema["db_id"]
    table_names = db_schema["table_names_original"]
    col_names = db_schema["column_names_original"]
    col_types = db_schema["column_types"]
    primary_keys = set(db_schema.get("primary_keys", []))
    foreign_keys = db_schema.get("foreign_keys", [])

    tables_to_cols = defaultdict(list)
    for col_idx, (table_idx, col_name) in enumerate(col_names):
        if table_idx == -1:
            continue
        col_type = col_types[col_idx] if col_idx < len(col_types) else "text"
        pk_tag = " (PRIMARY KEY)" if col_idx in primary_keys else ""
        tables_to_cols[table_idx].append(f"{col_name} {col_type}{pk_tag}")

    schema_lines = [f"Database Schema: {db_id}"]
    for table_idx, table_name in enumerate(table_names):
        cols_str = ", ".join(tables_to_cols[table_idx])
        schema_lines.append(f"Table {table_name}: ({cols_str})")

    if foreign_keys:
        fk_descriptions = []
        for src_col_idx, target_col_idx in foreign_keys:
            src_table_idx, src_col = col_names[src_col_idx]
            tgt_table_idx, tgt_col = col_names[target_col_idx]
            fk_descriptions.append(f"{table_names[src_table_idx]}.{src_col} = {table_names[tgt_table_idx]}.{tgt_col}")
        schema_lines.append("Foreign Keys: " + "; ".join(fk_descriptions))

    return "\n".join(schema_lines)

# Load schemas
with open("spider/tables.json", "r") as f:
    tables_data = json.load(f)
schemas = {db["db_id"]: format_schema(db) for db in tables_data}

# Load train examples
with open("spider/train_spider.json", "r") as f:
    train_raw = json.load(f)

# Balanced sampling across db_ids (target: 1500 examples)
TARGET_SUBSET_SIZE = 1500
db_to_examples = defaultdict(list)
for ex in train_raw:
    db_to_examples[ex["db_id"]].append(ex)

random.seed(42)
per_db_target = max(1, TARGET_SUBSET_SIZE // len(db_to_examples))
sampled, pool = [], []
for db_id, exs in db_to_examples.items():
    shuffled = list(exs)
    random.shuffle(shuffled)
    sampled.extend(shuffled[:per_db_target])
    pool.extend(shuffled[per_db_target:])

if len(sampled) < TARGET_SUBSET_SIZE:
    random.shuffle(pool)
    sampled.extend(pool[:(TARGET_SUBSET_SIZE - len(sampled))])

random.shuffle(sampled)

formatted_data = []
for item in sampled:
    db_id = item["db_id"]
    schema_text = schemas.get(db_id, f"Database Schema: {db_id}")
    formatted_data.append({
        "instruction": f"{schema_text}\n\nQuestion: {item['question']}",
        "output": item["query"],
        "db_id": db_id,
        "question": item["question"],
        "query": item["query"]
    })

# 90% train / 10% held-out eval
EVAL_SIZE = 150
held_out_eval = formatted_data[:EVAL_SIZE]
train_data = formatted_data[EVAL_SIZE:]

# Save to Drive
TRAIN_FILE = f"{DRIVE_DIR}/data/train_subset.json"
EVAL_FILE = f"{DRIVE_DIR}/data/held_out_eval.json"
with open(TRAIN_FILE, "w") as f:
    json.dump(train_data, f, indent=2)
with open(EVAL_FILE, "w") as f:
    json.dump(held_out_eval, f, indent=2)

print(f"Prepared {len(train_data)} train examples -> {TRAIN_FILE}")
print(f"Prepared {len(held_out_eval)} held-out eval examples -> {EVAL_FILE}")

Prepared 1350 train examples -> /content/drive/MyDrive/abdi/abdisql/data/train_subset.json
Prepared 150 held-out eval examples -> /content/drive/MyDrive/abdi/abdisql/data/held_out_eval.json


In [9]:
# 3.3 Compute & Save Zero-Shot Baseline Predictions on Held-Out Set
BASELINE_PREDS_FILE = f"{DRIVE_DIR}/eval_results/baseline_zero_shot_eval.json"

if not os.path.exists(BASELINE_PREDS_FILE):
    print(f"Generating baseline zero-shot predictions on held-out set ({len(held_out_eval)} items)...")
    baseline_results = []
    for idx, item in enumerate(held_out_eval):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": item["instruction"]}
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=128, do_sample=False)
        gen_sql = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

        baseline_results.append({
            "db_id": item["db_id"],
            "question": item["question"],
            "gold_sql": item["query"],
            "pred_sql": gen_sql
        })
        if (idx + 1) % 25 == 0:
            print(f"Progress: {idx + 1}/{len(held_out_eval)} baseline predictions generated.")

    with open(BASELINE_PREDS_FILE, "w") as f:
        json.dump(baseline_results, f, indent=2)
    print(f"Baseline outputs saved to {BASELINE_PREDS_FILE}")
else:
    print(f"Baseline outputs already exist at {BASELINE_PREDS_FILE}")

Baseline outputs already exist at /content/drive/MyDrive/abdi/abdisql/eval_results/baseline_zero_shot_eval.json


## Phase 4: QLoRA Fine-Tuning

Configured for Tesla T4 GPU:
- **LoRA rank:** 8, **alpha:** 16, **dropout:** 0.05
- **Target modules:** `q_proj`, `v_proj`
- **Batch size:** 1, **Gradient accumulation:** 8 (effective batch size = 8)
- **Learning rate:** 1e-4
- **Epochs:** 2
- **Gradient checkpointing:** ON
- **Checkpoints:** Saved every 100 steps to Google Drive

In [16]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from datasets import Dataset
import json
import torch
import gc

# 1. Cleanly reload base model to prevent corrupted PEFT wrappers
if 'model' in globals() and (isinstance(model, PeftModel) or hasattr(model, "peft_config")):
    del model
    gc.collect()
    torch.cuda.empty_cache()
    tokenizer, model = load_base_model(ACTIVE_MODEL_ID)

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.config.use_cache = False

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)

# 2. Purge ALL BFloat16 tensors (Parameters AND Buffers)
for param in model.parameters():
    if param.dtype == torch.bfloat16 or param.requires_grad:
        param.data = param.data.to(torch.float32)

for buffer in model.buffers():
    if buffer.dtype == torch.bfloat16:
        buffer.data = buffer.data.to(torch.float32)

model.print_trainable_parameters()

# 3. Format Dataset
def format_example_chatml(example):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["output"]}
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

with open(TRAIN_FILE, "r") as f:
    train_raw_list = json.load(f)
with open(EVAL_FILE, "r") as f:
    eval_raw_list = json.load(f)

train_dataset = Dataset.from_list([format_example_chatml(x) for x in train_raw_list])
eval_dataset = Dataset.from_list([format_example_chatml(x) for x in eval_raw_list])

gc.collect()
torch.cuda.empty_cache()

# 4. Train Config
try:
    from trl import SFTTrainer, SFTConfig
    sft_args = SFTConfig(
        output_dir=f"{DRIVE_DIR}/checkpoints",
        num_train_epochs=2,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=1e-4,
        weight_decay=0.01,
        fp16=False,
        bf16=False,
        gradient_checkpointing=False,
        logging_steps=20,
        save_steps=100,
        save_total_limit=2,
        eval_strategy="epoch",
        save_strategy="steps",
        optim="paged_adamw_8bit",
        dataset_text_field="text",
        max_length=1024,
        report_to="none"
    )
    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        args=sft_args,
        processing_class=tokenizer
    )
except Exception as e:
    from trl import SFTTrainer
    from transformers import TrainingArguments
    training_args = TrainingArguments(
        output_dir=f"{DRIVE_DIR}/checkpoints",
        num_train_epochs=2,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=1e-4,
        weight_decay=0.01,
        fp16=False,
        bf16=False,
        gradient_checkpointing=False,
        logging_steps=20,
        save_steps=100,
        save_total_limit=2,
        eval_strategy="epoch",
        save_strategy="steps",
        optim="paged_adamw_8bit",
        report_to="none"
    )
    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        max_length=1024,
        tokenizer=tokenizer,
        args=training_args
    )

print("Starting QLoRA fine-tuning...")
trainer.train()

FINAL_ADAPTER_DIR = f"{DRIVE_DIR}/final_adapter"
trainer.model.save_pretrained(FINAL_ADAPTER_DIR)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)
print(f"LoRA adapter weights saved to {FINAL_ADAPTER_DIR}")

Loading Qwen/Qwen2.5-Coder-1.5B-Instruct in 4-bit...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 1,089,536 || all params: 1,544,803,840 || trainable%: 0.0705


Adding EOS to train dataset:   0%|          | 0/1350 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1350 [00:00<?, ? examples/s]

...

Starting QLoRA fine-tuning...


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,0.826237,0.831047,0.827890,406854.000000,0.825309
2,0.745395,0.751840,0.798795,813708.000000,0.832141


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

LoRA adapter weights saved to /content/drive/MyDrive/abdi/abdisql/final_adapter


## Phase 5: Model Merging

Merge the trained LoRA adapter weights into the base model to create a single, standalone deployable model saved to Drive.

In [18]:
!pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 11.4 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [20]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch
import gc

# Safely free up training memory if variables exist
if 'model' in globals():
    del model
if 'trainer' in globals():
    del trainer
torch.cuda.empty_cache()
gc.collect()

print("Loading base model in FP16 for weight merging...")
base_model = AutoModelForCausalLM.from_pretrained(
    ACTIVE_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="cpu",  # Merge on CPU/RAM to avoid GPU OOM during weight fusion
    trust_remote_code=True
)

print("Loading trained LoRA adapter...")
peft_model = PeftModel.from_pretrained(base_model, FINAL_ADAPTER_DIR)

print("Merging adapter into base model weights...")
merged_model = peft_model.merge_and_unload()

MERGED_MODEL_DIR = f"{DRIVE_DIR}/merged_model"
merged_model.save_pretrained(MERGED_MODEL_DIR)

if 'tokenizer' not in globals():
    tokenizer = AutoTokenizer.from_pretrained(ACTIVE_MODEL_ID, trust_remote_code=True)
tokenizer.save_pretrained(MERGED_MODEL_DIR)

print(f"Merged model saved successfully to: {MERGED_MODEL_DIR}")

# Clean up RAM
del base_model
del peft_model
del merged_model
gc.collect()

Loading base model in FP16 for weight merging...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading trained LoRA adapter...


Merging adapter into base model weights...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model saved successfully to: /content/drive/MyDrive/abdi/abdisql/merged_model


757

## Phase 6: Evaluation & Execution Accuracy Gate

1. Generate SQL queries on the held-out set using the fine-tuned model.
2. Execute queries against real SQLite databases and calculate:
   - **Execution Accuracy:** % of queries where result sets match ground truth exactly.
   - **Syntax Validity Rate:** % of queries that run without SQLite errors.
3. Compare against Phase 2 zero-shot baseline.
4. **Gate Check:** Must show meaningful improvement over baseline before proceeding to export.
5. Post-gate validation on Spider's untouched `dev.json`.

In [21]:
import sqlite3
import re

# 6.1 Evaluation Engine
def extract_clean_sql(raw_text):
    text = raw_text.strip()
    match = re.search(r"```sql\s*(.*?)\s*```", text, re.DOTALL | re.IGNORECASE)
    if match: return match.group(1).strip().rstrip(";")
    match = re.search(r"```\s*(.*?)\s*```", text, re.DOTALL)
    if match: return match.group(1).strip().rstrip(";")
    return text.replace("<|im_end|>", "").strip().rstrip(";")

def normalize_val(val):
    if val is None: return None
    if isinstance(val, float): return round(val, 4)
    if isinstance(val, str):
        val_str = val.strip().lower()
        try: return round(float(val_str), 4)
        except ValueError: return val_str
    return val

def canonicalize_results(rows):
    canon = []
    for row in rows:
        canon.append(tuple(normalize_val(item) for item in row))
    return frozenset(canon)

def execute_sql(db_path, sql_query):
    clean_sql = extract_clean_sql(sql_query)
    try:
        conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True, timeout=3.0)
        cursor = conn.cursor()
        cursor.execute(clean_sql)
        rows = cursor.fetchall()
        conn.close()
        return True, rows, None
    except Exception as e:
        return False, None, str(e)

def evaluate_dataset(predictions_list, db_dir="spider/database"):
    correct = 0
    syntax_valid = 0
    total = len(predictions_list)

    for item in predictions_list:
        db_path = f"{db_dir}/{item['db_id']}/{item['db_id']}.sqlite"
        gold_ok, gold_res, _ = execute_sql(db_path, item["gold_sql"])
        pred_ok, pred_res, _ = execute_sql(db_path, item["pred_sql"])

        if pred_ok:
            syntax_valid += 1
            if gold_ok and (canonicalize_results(pred_res) == canonicalize_results(gold_res)):
                correct += 1

    return {
        "total": total,
        "execution_accuracy_pct": round((correct / total) * 100.0, 2),
        "syntax_validity_rate_pct": round((syntax_valid / total) * 100.0, 2),
        "correct": correct,
        "syntax_valid": syntax_valid
    }


In [22]:
# 6.2 Generate Fine-Tuned Predictions on Held-Out Set
print("Loading merged model for evaluation...")
eval_tokenizer = AutoTokenizer.from_pretrained(MERGED_MODEL_DIR)
eval_model = AutoModelForCausalLM.from_pretrained(
    MERGED_MODEL_DIR,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

print(f"Generating predictions for {len(held_out_eval)} held-out examples...")
ft_predictions = []
for idx, item in enumerate(held_out_eval):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": item["instruction"]}
    ]
    prompt = eval_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = eval_tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = eval_model.generate(**inputs, max_new_tokens=128, do_sample=False)
    gen_sql = eval_tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    ft_predictions.append({
        "db_id": item["db_id"],
        "question": item["question"],
        "gold_sql": item["query"],
        "pred_sql": gen_sql
    })

# 6.3 Run Execution Accuracy Benchmark
ft_metrics = evaluate_dataset(ft_predictions)
with open(BASELINE_PREDS_FILE, "r") as f:
    baseline_preds = json.load(f)
baseline_metrics = evaluate_dataset(baseline_preds)

print("\n================ EVALUATION BENCHMARK RESULTS ================")
print(f"Baseline Zero-Shot Accuracy:   {baseline_metrics['execution_accuracy_pct']}%")
print(f"Fine-Tuned Execution Accuracy: {ft_metrics['execution_accuracy_pct']}%")
print(f"Accuracy Gain:                 +{ft_metrics['execution_accuracy_pct'] - baseline_metrics['execution_accuracy_pct']:.2f}%")
print(f"Syntax Validity Rate:          {ft_metrics['syntax_validity_rate_pct']}%")
print("===============================================================")

# Save Results to Drive
results_payload = {
    "fine_tuned": ft_metrics,
    "baseline": baseline_metrics,
    "gain_pct": round(ft_metrics['execution_accuracy_pct'] - baseline_metrics['execution_accuracy_pct'], 2)
}
with open(f"{DRIVE_DIR}/eval_results/held_out_benchmark.json", "w") as f:
    json.dump(results_payload, f, indent=2)

# 6.4 Gate Check Assertion
assert ft_metrics['execution_accuracy_pct'] > baseline_metrics['execution_accuracy_pct'], \
    "GATE FAILURE: Fine-tuned model did not outperform zero-shot baseline! Review Phase 4 training hyperparameters."
print("Phase 6 Gate Check: PASSED. Proceeding to Spider dev set sanity check.")

Loading merged model for evaluation...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Generating predictions for 150 held-out examples...

================ EVALUATION BENCHMARK RESULTS ================
Baseline Zero-Shot Accuracy:   51.33%
Fine-Tuned Execution Accuracy: 54.67%
Accuracy Gain:                 +3.34%
Syntax Validity Rate:          69.33%
Phase 6 Gate Check: PASSED. Proceeding to Spider dev set sanity check.


In [23]:
# 6.5 Independent Confirmation on Spider Dev Set (Untouched until now)
print("Running post-gate validation on Spider dev set...")
with open("spider/dev.json", "r") as f:
    dev_raw = json.load(f)

# Sample 100 dev examples for fast validation
random.seed(42)
dev_sample = random.sample(dev_raw, 100)
dev_predictions = []

for item in dev_sample:
    db_id = item["db_id"]
    schema_text = schemas.get(db_id, f"Database Schema: {db_id}")
    prompt_text = f"{schema_text}\n\nQuestion: {item['question']}"
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt_text}
    ]
    prompt = eval_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = eval_tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = eval_model.generate(**inputs, max_new_tokens=128, do_sample=False)
    gen_sql = eval_tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    dev_predictions.append({
        "db_id": db_id,
        "gold_sql": item["query"],
        "pred_sql": gen_sql
    })

dev_metrics = evaluate_dataset(dev_predictions)
print(f"\nSpider Dev Set Execution Accuracy: {dev_metrics['execution_accuracy_pct']}%")
print(f"Spider Dev Set Syntax Validity:   {dev_metrics['syntax_validity_rate_pct']}%")

with open(f"{DRIVE_DIR}/eval_results/dev_set_benchmark.json", "w") as f:
    json.dump(dev_metrics, f, indent=2)

Running post-gate validation on Spider dev set...

Spider Dev Set Execution Accuracy: 47.0%
Spider Dev Set Syntax Validity:   62.0%


## Phase 7: Export & Quantization

Export instructions and artifacts for deployment:

In [24]:
# Documenting deployment export and results
results_md = f"""# AbdiSQL v1 — Build & Benchmark Results

## Model Information
- **Base Model:** {ACTIVE_MODEL_ID}
- **Dialect:** SQLite
- **Training Method:** QLoRA 4-bit (rank 8, alpha 16, lr=1e-4, 2 epochs)
- **Merged Weights:** `{MERGED_MODEL_DIR}`

## Benchmark Metrics
- **Held-Out Set Execution Accuracy (Fine-Tuned):** {ft_metrics['execution_accuracy_pct']}%
- **Held-Out Set Execution Accuracy (Zero-Shot Baseline):** {baseline_metrics['execution_accuracy_pct']}%
- **Gain over Baseline:** +{ft_metrics['execution_accuracy_pct'] - baseline_metrics['execution_accuracy_pct']:.2f}%
- **Syntax Validity Rate:** {ft_metrics['syntax_validity_rate_pct']}%
- **Spider Dev Set Execution Accuracy:** {dev_metrics['execution_accuracy_pct']}%

## Quantization / Edge Deployment Instructions
To convert the merged weights to GGUF format for llama.cpp / edge inference:
```bash
git clone https://github.com/ggerganov/llama.cpp.git
cd llama.cpp
pip install -r requirements.txt
python convert_hf_to_gguf.py {MERGED_MODEL_DIR} --outfile abdisql-q4_k_m.gguf --outtype q4_k_m
```
"""

with open(f"{DRIVE_DIR}/RESULTS.md", "w") as f:
    f.write(results_md)
print(f"RESULTS.md created at {DRIVE_DIR}/RESULTS.md")
print(results_md)

RESULTS.md created at /content/drive/MyDrive/abdi/abdisql/RESULTS.md
# AbdiSQL v1 — Build & Benchmark Results

## Model Information
- **Base Model:** Qwen/Qwen2.5-Coder-1.5B-Instruct
- **Dialect:** SQLite
- **Training Method:** QLoRA 4-bit (rank 8, alpha 16, lr=1e-4, 2 epochs)
- **Merged Weights:** `/content/drive/MyDrive/abdi/abdisql/merged_model`

## Benchmark Metrics
- **Held-Out Set Execution Accuracy (Fine-Tuned):** 54.67%
- **Held-Out Set Execution Accuracy (Zero-Shot Baseline):** 51.33%
- **Gain over Baseline:** +3.34%
- **Syntax Validity Rate:** 69.33%
- **Spider Dev Set Execution Accuracy:** 47.0%

## Quantization / Edge Deployment Instructions
To convert the merged weights to GGUF format for llama.cpp / edge inference:
```bash
git clone https://github.com/ggerganov/llama.cpp.git
cd llama.cpp
pip install -r requirements.txt
python convert_hf_to_gguf.py /content/drive/MyDrive/abdi/abdisql/merged_model --outfile abdisql-q4_k_m.gguf --outtype q4_k_m
```

